---

# Plota Mapa Diário de Flashes do GLM do GOES-16/19.

---

- `OBJETIVO`:
> Este código plota o mapa de acumulado de flashes para um determinado dia. Ele extrai também a quantidade de flashes ocorrido naquele dia para alguns Estados brasileiros.


- `DADOS DE ENTRADA`:
> Arquivo NETCDF de flashes acumulados para um determinado dia. Exemplo de nome do arquivo: `flash_glm_goes_2020-06-30.nc`


- `DADOS DE SAÍDA`:
> 1. Figura JPG de acumulado de flashes para um determinado dia. Exemplo: `GLM_03_flash_goes16_2020-06-30.jpg`


- `OBSERVAÇÕES`:
   > Sem observações

- `REALIZADO POR`:
> Enrique V. Mattos - 03/10/2025

- `ATUALIZADO POR`:
> Enrique V. Mattos - 27/04/2026
---


# **1° Passo:** Preparando Ambiente

In [ ]:
# instalações
!pip install -q ultraplot cartopy salem rasterio pyproj geopandas

# monta o drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/000_CODIGOS_REFERENCIA/03_RELAMPAGOS_SATELITE_REFERENCIA'

# diretório de entrada
dir_input = f'{dir}/output/glm_diario_goes16'

# diretório de saída
dir_output = f'{dir}/output/glm_figuras_goes16'

# importa bibliotecas
import ultraplot as uplt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
import os
import time
import salem
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Plota Figura

In [ ]:
%%time
#==================================================================================================#
#                                   DEFINIÇÃO DOS LIMITES DA IMAGEM
#==================================================================================================#
# limites das latitudes e longitudes
lonmin, lonmax, latmin, latmax = -90, -30, -40, 10

# extensão da imagem [min. lon, min. lat, max. lon, max. lat]
extent = [lonmin, latmin, lonmax, latmax]

#==================================================================================================#
#                                 LEITURA DO ARQUIVO NETCDF DIÁRIO
#==================================================================================================#
# data
ano, mes, dia = '2020', '06', '30'

# leitura do arquivo
glm_amazon_diario = xr.open_dataset(f'{dir_input}/flash_glm_goes_2020-06-30.nc').sel(lon=slice(lonmin, lonmax), lat=slice(latmin, latmax))

#==================================================================================================#
#                                 DEFINIÇÕES DO GRÁFICO
#==================================================================================================#
# cria moldura da figura
fig, ax = uplt.subplots(axheight=6.9, axwidth=6.8, tight=True, proj='pcarree')

# formata os eixos
ax.format(coast=True, borders=True, innerborders=False,
          labels=True, latlines=20, lonlines=20,
          latlim=(latmin, latmax), lonlim=(lonmin, lonmax),
          small='20px', large='25px',
          title=f'Acumulado de Relâmpagos \n',
          titleloc='l',
          titleweight='bold',
          titlecolor='bright red')

# plota subtítulo
ax.text(0.000, 1.035,
        f'Satélite: GOES-16 (8km) | Sensor: GLM | Período: {ano}-{mes}-{dia}',
        transform = ax.transAxes,
        color = 'gray',
        fontsize = 9,
        verticalalignment = 'top')

#==================================================================================================#
#                                       PLOTA FIGURA
#==================================================================================================#
# quantidade de flashes por dia para a área TOTAL da imagem plotada
total = int(glm_amazon_diario['flash'].sum(('lat', 'lon')))

# substituir valores "zero" por "NaN"
glm_amazon_diario = glm_amazon_diario.where(glm_amazon_diario != 0, np.nan)

# figura
# aqui dividimos a matriz por 64 para termos a unidade em km2. Lembre-se que
# a matriz de flashes tem tamanho de 8 km x 8 km (ou seja, área de 64 km2).
map1 = ax.contourf(glm_amazon_diario['lon'],
                   glm_amazon_diario['lat'],
                   glm_amazon_diario['flash']/64.,
                   cmap='jet',
                   levels=uplt.arange(0, 5, 1),
                   extend='max')

# total de relâmpagos
ax.text(lonmax-22., latmin+0.9, f'Total: {int(total)} relâmpagos', color='red', fontsize=15)

# plota contornos dos Estados
shapefile = list(shpreader.Reader('https://github.com/evmpython/Minicurso_UFMS_SEMADESC_marco_2026/raw/main/01_utils/BR_UF_2019.shp').geometries())
ax.add_geometries(shapefile, ccrs.PlateCarree(), edgecolor='grey', facecolor='none', linewidth=0.5, zorder=2)

# barra de cores
fig.colorbar(map1, loc='b', label='relâmpagos/$km^{2}$', ticks=1, ticklabelsize=15, labelsize=15, space=-1.3, width=0.3)

# salva figura
fig.save(f'{dir_output}/GLM_03_flash_goes16_{ano}-{mes}-{dia}.jpg', dpi=300)

In [ ]:
# mostra os dados
glm_amazon_diario

# Contabiliza os Flashes por Estado do Brasil


In [ ]:
%%time
# lendo shapefile dos estados
sp = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/UFs/SP/SP_UF_2019.shp')
pr = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/UFs/PR/PR_UF_2019.shp')
sc = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/UFs/SC/SC_UF_2019.shp')
rs = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/UFs/RS/RS_UF_2019.shp')

# estraindo os flashes de cada estado
flash_sp = glm_amazon_diario.salem.roi(shape=sp).sum(('lat', 'lon'))
flash_pr = glm_amazon_diario.salem.roi(shape=pr).sum(('lat', 'lon'))
flash_sc = glm_amazon_diario.salem.roi(shape=sc).sum(('lat', 'lon'))
flash_rs = glm_amazon_diario.salem.roi(shape=rs).sum(('lat', 'lon'))

# imprime na tela a quantidade de relâmpagos de cada estado
print('Total de relâmpagos em SP =', int(flash_sp['flash'].item()))
print('Total de relâmpagos no PR =', int(flash_pr['flash'].item()))
print('Total de relâmpagos no SC =', int(flash_sc['flash'].item()))
print('Total de relâmpagos no RS =', int(flash_rs['flash'].item()))